# T3000 Wagon Brake Mechanism Animation
This notebook generates an animated GIF comparing **Normal Braking** vs **Manual Brake Pre-Applied** for the T3000 wagon pneumatic brake system.

### Key concept
- **Normal braking**: piston travels full free stroke → cylinder volume increases → pressure builds slowly after contact
- **Manual brake pre-applied**: pad already in contact → cylinder at max volume from the start → higher pressure gradient, faster max pressure


In [ ]:
# Install required library
!pip install Pillow --quiet

In [ ]:
from PIL import Image, ImageDraw
import math

# ── Canvas ──────────────────────────────────────────────────────────────────
W, H = 880, 460

# ── Colours ─────────────────────────────────────────────────────────────────
BG             = (30,  30,  40)
WHEEL_COLOR    = (180, 180, 200)
WHEEL_RIM      = (220, 220, 240)
PAD_COLOR      = (220, 120, 50)
CYLINDER_COLOR = (80,  100, 130)
PISTON_COLOR   = (100, 160, 200)
MANUAL_ROD_COLOR = (200, 180, 60)
DIM_TEXT       = (150, 150, 170)
GREEN          = (80,  210, 110)
RED            = (220, 80,  80)
BLUE           = (80,  160, 230)
ORANGE         = (230, 140, 50)

print("Imports and colour constants ready.")

In [ ]:
# ── Wheel ────────────────────────────────────────────────────────────────────
WHEEL_CX = 680
WHEEL_CY = 230
WHEEL_R  = 90

# ── Brake Cylinder ───────────────────────────────────────────────────────────
# Made long enough so the piston (P) never exits the cylinder
CYL_LEFT  = 60
CYL_TOP   = 185
CYL_H     = 90
CYL_MID_Y = CYL_TOP + CYL_H // 2   # vertical centre = 230
PISTON_W  = 28

# ── Linkage ──────────────────────────────────────────────────────────────────
WHEEL_LEFT_X = WHEEL_CX - WHEEL_R   # left surface of wheel = 590
ROD_LEN      = 80                   # rod: piston right face → pad left face
PAD_W        = 18
PAD_H        = 64

# ── Key piston positions ──────────────────────────────────────────────────────
# Contact: pad right face just touches wheel left surface
CONTACT_PISTON_RIGHT = WHEEL_LEFT_X - PAD_W - ROD_LEN   # 492
# Rest: piston retracted inside cylinder (full free-stroke gap open)
REST_PISTON_RIGHT    = CONTACT_PISTON_RIGHT - 120        # 372
# Cylinder right inner wall (piston never goes past this)
CYL_RIGHT = CONTACT_PISTON_RIGHT + PISTON_W + 20         # 540
CYL_W     = CYL_RIGHT - CYL_LEFT                         # 480

print(f"Cylinder width : {CYL_W} px")
print(f"Rest piston_right  : {REST_PISTON_RIGHT}")
print(f"Contact piston_right: {CONTACT_PISTON_RIGHT}")
print(f"Cylinder right wall : {CYL_RIGHT}")

In [ ]:
def draw_frame(mode: str, t: float, pressure_t: float) -> Image.Image:
    """
    Draw a single animation frame.

    Parameters
    ----------
    mode       : 'normal'  – piston travels free stroke before contact
                 'manual'  – pad already at wheel, piston stationary
    t          : 0‒1  piston travel progress  (1 = pad touching wheel)
    pressure_t : 0‒1  pressure build-up progress
    """
    img  = Image.new("RGB", (W, H), BG)
    draw = ImageDraw.Draw(img)

    # ── Title bar ────────────────────────────────────────────────────────────
    title       = "NORMAL BRAKING" if mode == "normal" else "MANUAL BRAKE PRE-APPLIED"
    title_color = BLUE             if mode == "normal" else ORANGE
    draw.rectangle([0, 0, W, 38], fill=(18, 18, 28))
    draw.text((W // 2, 19), title, fill=title_color, anchor="mm")

    # ── Wheel ────────────────────────────────────────────────────────────────
    draw.ellipse([WHEEL_CX - WHEEL_R, WHEEL_CY - WHEEL_R,
                  WHEEL_CX + WHEEL_R, WHEEL_CY + WHEEL_R],
                 fill=WHEEL_COLOR, outline=WHEEL_RIM, width=4)
    draw.ellipse([WHEEL_CX - 18, WHEEL_CY - 18,
                  WHEEL_CX + 18, WHEEL_CY + 18],
                 fill=(55, 55, 75), outline=WHEEL_RIM, width=2)
    for deg in range(0, 360, 45):
        a = math.radians(deg)
        draw.line([WHEEL_CX + 18*math.cos(a), WHEEL_CY + 18*math.sin(a),
                   WHEEL_CX + WHEEL_R*math.cos(a), WHEEL_CY + WHEEL_R*math.sin(a)],
                  fill=WHEEL_RIM, width=2)
    draw.text((WHEEL_CX, WHEEL_CY + WHEEL_R + 16), "WHEEL", fill=DIM_TEXT, anchor="mm")

    # ── Piston position ──────────────────────────────────────────────────────
    if mode == "normal":
        # Travel from rest → contact over t
        piston_right = REST_PISTON_RIGHT + (CONTACT_PISTON_RIGHT - REST_PISTON_RIGHT) * t
    else:
        # Manual: pad already at wheel — piston stays at contact position
        piston_right = CONTACT_PISTON_RIGHT

    piston_left = piston_right - PISTON_W
    pad_left    = piston_right + ROD_LEN
    pad_right   = pad_left    + PAD_W

    # ── Air chamber ──────────────────────────────────────────────────────────
    air_width = max(0, piston_left - CYL_LEFT - 4)

    # ── Cylinder body ────────────────────────────────────────────────────────
    draw.rounded_rectangle([CYL_LEFT, CYL_TOP, CYL_RIGHT, CYL_TOP + CYL_H],
                            radius=6, fill=CYLINDER_COLOR,
                            outline=(120, 140, 175), width=3)
    draw.text((CYL_LEFT + CYL_W // 2, CYL_TOP - 18),
              "BRAKE CYLINDER", fill=DIM_TEXT, anchor="mm")

    # Air fill — colour intensifies with pressure
    if air_width > 0:
        air_col = (int(60 + 40*pressure_t),
                   int(120 + 60*pressure_t),
                   int(200 + 40*pressure_t))
        draw.rectangle([CYL_LEFT + 4, CYL_TOP + 4,
                        CYL_LEFT + 4 + air_width, CYL_TOP + CYL_H - 4],
                       fill=air_col)
        if air_width > 40:
            draw.text((CYL_LEFT + 4 + air_width // 2, CYL_MID_Y),
                      "AIR", fill=(210, 235, 255), anchor="mm")

    # ── Piston ───────────────────────────────────────────────────────────────
    draw.rectangle([piston_left, CYL_TOP + 6, piston_right, CYL_TOP + CYL_H - 6],
                   fill=PISTON_COLOR, outline=(160, 210, 235), width=2)
    draw.text(((piston_left + piston_right) // 2, CYL_MID_Y),
              "P", fill=(20, 20, 40), anchor="mm")

    # ── Connecting rod ───────────────────────────────────────────────────────
    draw.line([piston_right, CYL_MID_Y, pad_left, CYL_MID_Y],
              fill=(160, 170, 190), width=5)

    # ── Brake pad ────────────────────────────────────────────────────────────
    in_contact = pad_right >= WHEEL_LEFT_X - 1
    draw.rectangle([pad_left,  CYL_MID_Y - PAD_H // 2,
                    pad_right, CYL_MID_Y + PAD_H // 2],
                   fill=PAD_COLOR if in_contact else (140, 90, 35),
                   outline=(240, 160, 60), width=2)

    # ── Manual brake rod (manual mode only) ──────────────────────────────────
    if mode == "manual":
        mb_top_y = CYL_TOP - 32
        cx = (pad_left + pad_right) // 2
        draw.line([cx, CYL_MID_Y - PAD_H // 2, cx, mb_top_y],
                  fill=MANUAL_ROD_COLOR, width=4)
        draw.rounded_rectangle([cx - 38, mb_top_y - 14, cx + 38, mb_top_y],
                                radius=4, fill=MANUAL_ROD_COLOR,
                                outline=(240, 210, 80), width=2)
        draw.text((cx, mb_top_y - 24), "MANUAL BRAKE ROD",
                  fill=MANUAL_ROD_COLOR, anchor="mm")

    # ── Contact badge ────────────────────────────────────────────────────────
    draw.text((WHEEL_CX, 60),
              "● CONTACT" if in_contact else "○ NO CONTACT",
              fill=GREEN if in_contact else RED, anchor="mm")

    # ── Free-stroke gap arrow (normal mode only) ─────────────────────────────
    if mode == "normal":
        gap = WHEEL_LEFT_X - pad_right
        if gap > 6:
            ay = CYL_MID_Y + PAD_H // 2 + 22
            draw.line([pad_right, ay, WHEEL_LEFT_X, ay], fill=(180, 180, 100), width=2)
            draw.polygon([(pad_right,    ay), (pad_right + 8,    ay - 5), (pad_right + 8,    ay + 5)], fill=(180, 180, 100))
            draw.polygon([(WHEEL_LEFT_X, ay), (WHEEL_LEFT_X - 8, ay - 5), (WHEEL_LEFT_X - 8, ay + 5)], fill=(180, 180, 100))
            draw.text(((pad_right + WHEEL_LEFT_X) // 2, ay + 14),
                      "free stroke gap", fill=(180, 180, 100), anchor="mm")

    # ── Pressure gauge ───────────────────────────────────────────────────────
    g_cx, g_cy, g_r = W - 72, 110, 48
    draw.ellipse([g_cx - g_r, g_cy - g_r, g_cx + g_r, g_cy + g_r],
                 fill=(40, 40, 55), outline=(120, 130, 150), width=2)
    arc_end = 210 + int(240 * pressure_t)
    if arc_end > 210:
        draw.arc([g_cx - g_r + 6, g_cy - g_r + 6,
                  g_cx + g_r - 6, g_cy + g_r - 6],
                 start=210, end=arc_end,
                 fill=BLUE if mode == "normal" else ORANGE, width=7)
    na = math.radians(210 + 240 * pressure_t)
    draw.line([g_cx, g_cy,
               g_cx + (g_r - 12)*math.cos(na),
               g_cy + (g_r - 12)*math.sin(na)],
              fill=(230, 230, 230), width=2)
    draw.ellipse([g_cx - 4, g_cy - 4, g_cx + 4, g_cy + 4], fill=(200, 200, 200))
    draw.text((g_cx, g_cy + g_r + 12),
              f"Pressure {int(pressure_t * 100)}%", fill=DIM_TEXT, anchor="mm")

    # ── Volume / gradient label ───────────────────────────────────────────────
    if mode == "normal":
        vol_txt = "Cylinder Vol: INCREASING  →  LOWER pressure gradient"
        vol_col = BLUE
    else:
        vol_txt = "Cylinder Vol: MAX (constant)  →  HIGHER pressure gradient"
        vol_col = ORANGE
    draw.text((W // 2 - 40, H - 40), vol_txt, fill=vol_col, anchor="mm")

    # ── Phase label ──────────────────────────────────────────────────────────
    if mode == "normal":
        phase = "Piston free-stroking — volume expanding" if t < 0.95 else "Pad contacts wheel — force builds"
    else:
        phase = "Pad at wheel from start — immediate force build-up, faster max pressure"
    draw.text((W // 2 - 40, H - 18), phase, fill=DIM_TEXT, anchor="mm")

    # ── Progress bar ─────────────────────────────────────────────────────────
    bx0, by0, bx1, by1 = 60, H - 60, W - 160, H - 50
    draw.rectangle([bx0, by0, bx1, by1], fill=(50, 50, 65), outline=(80, 80, 100))
    prog = bx0 + int((bx1 - bx0) * (t if mode == "normal" else pressure_t))
    draw.rectangle([bx0, by0, prog, by1],
                   fill=BLUE if mode == "normal" else ORANGE)

    return img

print("draw_frame() defined.")

In [ ]:
def ease_in_out(t: float) -> float:
    """Smooth-step easing (cubic Hermite)."""
    return t * t * (3 - 2 * t)


N    = 48   # animation frames per mode
HOLD = 14   # hold frames at start/end of each mode

frames = []

# ── Normal braking ────────────────────────────────────────────────────────────
# Pressure only starts building after pad contacts wheel (~t > 0.85)
for i in range(N):
    t = ease_in_out(i / (N - 1))
    p = ease_in_out(max(0, (t - 0.85) / 0.15)) if t > 0.85 else 0.0
    frames.append(draw_frame("normal", t, p))
for _ in range(HOLD):
    frames.append(draw_frame("normal", 1.0, 1.0))
for _ in range(HOLD):
    frames.append(draw_frame("normal", 0.0, 0.0))

# ── Manual brake pre-applied ──────────────────────────────────────────────────
# Piston stationary (t=1 always); pressure builds immediately from frame 0
for i in range(N):
    p = ease_in_out(i / (N - 1))
    frames.append(draw_frame("manual", 1.0, p))
for _ in range(HOLD):
    frames.append(draw_frame("manual", 1.0, 1.0))
for _ in range(HOLD):
    frames.append(draw_frame("manual", 1.0, 0.0))

print(f"Total frames generated: {len(frames)}")

In [ ]:
OUTPUT_PATH = "brake_mechanism.gif"

frames[0].save(
    OUTPUT_PATH,
    save_all=True,
    append_images=frames[1:],
    duration=65,    # ms per frame
    loop=0          # loop forever
)

print(f"GIF saved → {OUTPUT_PATH}")

In [ ]:
# Preview a sample frame inline
from IPython.display import display

display(draw_frame("normal", 0.0, 0.0))   # normal mode — rest position
display(draw_frame("manual", 1.0, 0.5))   # manual mode — mid pressure build-up